In [ ]:
import re 
from rdkit import Chem
from collections import Counter
import random 
import pandas as pd 
from tqdm import tqdm

In [ ]:
train_data = pd.read_csv('./data/train_data/original/train.csv')
val_data = pd.read_csv('./data/train_data/original/val.csv')

In [ ]:
def fill_source_peptide(source: str, target: str) -> str:
    source = source.split('|')
    target_merge = target.split('|')
    indices = [idx for idx, s in enumerate(source) if s == '?']
    mask_count = source.count('?')
    new_target = []
    m = 0
    if len(target_merge) == mask_count:
        for idx, s in enumerate(source):
            if idx in indices:
                t = target_merge[m]
                new_target.append(t)
                m += 1
            else:
                new_target.append(s)
        new_target = [aa for aa in new_target[:-1]] + [new_target[-1]]
        new_target = '|'.join(new_target)
    else:
        new_target = 'none'
    return new_target

In [ ]:
train_filled_chuckles = train_data.apply(lambda row: fill_source_peptide(row['Source_Mol'], row['Target_Mol']), axis=1).tolist()
val_filled_chuckles = val_data.apply(lambda row: fill_source_peptide(row['Source_Mol'], row['Target_Mol']), axis=1).tolist()

## Generate Mask

In [ ]:
def mask_chuckles(chuckles: str):
    masked_monomers = []
    monomers = chuckles.split('|')

    num_mask = round(random.triangular(1, len(monomers)*0.4, 0))
    mask_index = random.sample(range(0, len(monomers)), num_mask) 

    for i in mask_index: 
        masked_monomers.append(monomers[i])
        monomers[i] = '?'

    return '|'.join(monomers), '|'.join(masked_monomers)

def shift_chuckles(chuckles: str):
    def _extract_unpaired_num(sequence_part: str) -> str:
        ring_numbers = re.findall(r'%\d+|\d', sequence_part)
        counts = Counter(ring_numbers)
        for ring_num, count in counts.items():
            if count % 2 != 0:  # Odd count indicates unpaired ring
                return ring_num
        return ''
    
    chuckles_list = chuckles.split('|')
    shift = random.randint(0, len(chuckles_list)-1)
    
    first_number = _extract_unpaired_num(chuckles_list[0])
    last_number = _extract_unpaired_num(chuckles_list[-1])
    
    if first_number != last_number:
        first_number, last_number = '', ''
    
    chuckles_list[0] = chuckles_list[0].replace(first_number, '')
    chuckles_list[-1] = chuckles_list[-1].replace(first_number, '')
    
    shift = shift % len(chuckles_list)
    chuckles_shifted = chuckles_list[-shift:] + chuckles_list[:-shift]
    
    if first_number:
        chuckles_shifted[0] = (chuckles_shifted[0][:1] + 
                            first_number + 
                            chuckles_shifted[0][1:])
        
        chuckles_shifted[-1] = re.sub(r'C\(=O\)$', 
                                    f'C{last_number}(=O)', 
                                    chuckles_shifted[-1])
    
    combined_smiles = ''.join(chuckles_shifted)
    if Chem.MolFromSmiles(combined_smiles):
        return '|'.join(chuckles_shifted)
    else:

        return chuckles

In [ ]:
random.seed(109)

train, val1, val2 = [], [], []

for chuckles in train_filled_chuckles: 
    train.append(mask_chuckles(chuckles))

for chuckles in val_filled_chuckles: 
    val1.append(mask_chuckles(chuckles))

for chuckles in tqdm(val_filled_chuckles):
    shifted_chuckles = shift_chuckles(chuckles) 
    val2.append(mask_chuckles(shifted_chuckles))

In [ ]:
train_source, train_target = [], [] 
val1_source, val1_target, val2_source, val2_target = [], [], [], []

for source, target in train: 
    train_source.append(source), train_target.append(target)

for source, target in val1: 
    val1_source.append(source), val1_target.append(target)

for source, target in val2: 
    val2_source.append(source), val2_target.append(target)

In [ ]:
train_df = pd.DataFrame({'Source_Mol': train_source, 'Target_Mol': train_target})
val1_df = pd.DataFrame({'Source_Mol': val1_source, 'Target_Mol': val1_target})
val2_df = pd.DataFrame({'Source_Mol': val2_source, 'Target_Mol': val2_target})

In [ ]:
train_df.to_csv('./data/train_data/static/train.csv', index=False)

val1_df.to_csv('./data/train_data/static/val1.csv', index=False)
val2_df.to_csv('./data/train_data/static/val2.csv', index=False)